## Inicialización y Carga de Datos

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, regexp_replace, round

In [2]:
# Inicializar sesión
spark = SparkSession.builder \
    .appName("ManipulacionDataFrames_Bio") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 10:13:35 WARN Utils: Your hostname, MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.93 instead (on interface en0)
26/04/13 10:13:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 10:13:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Cargar links (interacciones)
df_links = spark.read.option("header", "true").option("sep", " ").option("inferSchema", "true") \
    .csv("../data/9606.protein.links.v12.0.txt.gz")

In [4]:
# Cargar info (metadatos de proteínas)
df_info = spark.read.option("header", "true").option("sep", "\t").option("inferSchema", "true") \
    .csv("../data/9606.protein.info.v12.0.txt.gz") \
    .withColumnRenamed("#string_protein_id", "protein_id")

In [5]:
display(df_links.head(3))
display(df_info.head(3))

[Row(protein1='9606.ENSP00000000233', protein2='9606.ENSP00000356607', combined_score=173),
 Row(protein1='9606.ENSP00000000233', protein2='9606.ENSP00000427567', combined_score=154),
 Row(protein1='9606.ENSP00000000233', protein2='9606.ENSP00000253413', combined_score=151)]

[Row(protein_id='9606.ENSP00000000233', preferred_name='ARF5', protein_size=180, annotation='ADP-ribosylation factor 5; GTP-binding protein involved in protein trafficking; may modulate vesicle budding and uncoating within the Golgi apparatus. Belongs to the small GTPase superfamily. Arf family.'),
 Row(protein_id='9606.ENSP00000000412', preferred_name='M6PR', protein_size=277, annotation='Cation-dependent mannose-6-phosphate receptor; Transport of phosphorylated lysosomal enzymes from the Golgi complex and the cell surface to lysosomes. Lysosomal enzymes bearing phosphomannosyl residues bind specifically to mannose-6-phosphate receptors in the Golgi apparatus and the resulting receptor-ligand complex is transported to an acidic prelyosomal compartment where the low pH mediates the dissociation of the complex.'),
 Row(protein_id='9606.ENSP00000001008', preferred_name='FKBP4', protein_size=459, annotation='Peptidyl-prolyl cis-trans isomerase FKBP4, N-terminally processed; Immunophilin p

## Modificación de Datos

In [6]:
# Modificar datos: Quitar el prefijo "9606."
df_info_clean = df_info.withColumn(
    "protein_id", 
    regexp_replace(col("protein_id"), "9606.", "")
)

In [7]:
# Limpiar los links para que coincidan en el Join posterior
df_links_clean = df_links.withColumn("protein1", regexp_replace(col("protein1"), "9606.", ""))

In [8]:
df_info_clean.select("protein_id", "preferred_name").show(5)

+---------------+--------------+
|     protein_id|preferred_name|
+---------------+--------------+
|ENSP00000000233|          ARF5|
|ENSP00000000412|          M6PR|
|ENSP00000001008|         FKBP4|
|ENSP00000001146|       CYP26B1|
|ENSP00000002125|       NDUFAF7|
+---------------+--------------+
only showing top 5 rows


## Agregar Nuevas Columnas Calculadas

In [9]:
# 1. Columna calculada simple: Score de 0 a 1
# 2. Columna condicional (Modificación lógica): Categorización
df_extended = df_links_clean.withColumn("score_decimal", round(col("combined_score") / 1000, 2)) \
    .withColumn("confidence_level", 
        when(col("combined_score") >= 900, "Highest")
        .when(col("combined_score") >= 700, "High")
        .otherwise("Medium/Low")
    )

df_extended.select("protein1", "combined_score", "score_decimal", "confidence_level").show(5)

+---------------+--------------+-------------+----------------+
|       protein1|combined_score|score_decimal|confidence_level|
+---------------+--------------+-------------+----------------+
|ENSP00000000233|           173|         0.17|      Medium/Low|
|ENSP00000000233|           154|         0.15|      Medium/Low|
|ENSP00000000233|           151|         0.15|      Medium/Low|
|ENSP00000000233|           471|         0.47|      Medium/Low|
|ENSP00000000233|           201|          0.2|      Medium/Low|
+---------------+--------------+-------------+----------------+
only showing top 5 rows


## Filtrado

In [10]:
# Filtrar: Interacciones de "Highest" confianza que tengan un score decimal específico
# y que no involucren a la misma proteína (auto-interacción, si existiera)
df_filtered = df_extended.filter(
    (col("confidence_level") == "Highest") & 
    (col("score_decimal") > 0.95) &
    (col("protein1") != col("protein2"))
)

In [11]:
print(f"Total de interacciones críticas encontradas: {df_filtered.count()}")
df_filtered.show(10)

Total de interacciones críticas encontradas: 110298
+---------------+--------------------+--------------+-------------+----------------+
|       protein1|            protein2|combined_score|score_decimal|confidence_level|
+---------------+--------------------+--------------+-------------+----------------+
|ENSP00000000412|9606.ENSP00000438085|           993|         0.99|         Highest|
|ENSP00000000412|9606.ENSP00000371175|           957|         0.96|         Highest|
|ENSP00000000412|9606.ENSP00000349437|           991|         0.99|         Highest|
|ENSP00000001008|9606.ENSP00000351646|           979|         0.98|         Highest|
|ENSP00000001008|9606.ENSP00000444810|           972|         0.97|         Highest|
|ENSP00000001008|9606.ENSP00000359385|           983|         0.98|         Highest|
|ENSP00000001008|9606.ENSP00000354558|           961|         0.96|         Highest|
|ENSP00000001008|9606.ENSP00000482075|           990|         0.99|         Highest|
|ENSP00000001

In [12]:
spark.stop()